# LUMAQ: Machine Learning for Solar Battery Runtime Prediction

**DSN 2026 AI Bootcamp Hackathon — Machine Learning with Python**

LUMAQ is an intelligent energy guidance system developed by Ayoola Timilehin Israel. It estimates the remaining runtime of a solar battery system, detects unusual consumption and converts predictions into direct energy-use messages.

The notebook uses reproducible **simulated proof-of-concept data** based on a 48 V solar system. The records do not represent measurements exported from the physical LUMAQ prototype.

## 1. Problem and objective

Many solar and inverter interfaces display electrical measurements without explaining how long the available energy will last. LUMAQ models remaining runtime from battery state, connected load, solar input and operating conditions. A separate anomaly model identifies consumption patterns that require inspection.

In [ ]:
from pathlib import Path
import sys
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

local_candidates = [Path('../src'), Path('src')]
module_directory = next((path for path in local_candidates if (path / 'lumaq_ml.py').exists()), None)
if module_directory is None:
    module_directory = Path('/content/lumaq_src')
    module_directory.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/Elontim/LUMAQ-DSN-2026/main/src/lumaq_ml.py',
        module_directory / 'lumaq_ml.py'
    )
sys.path.insert(0, str(module_directory.resolve()))

from lumaq_ml import (
    FEATURES,
    TARGET,
    energy_guidance,
    fit_anomaly_model,
    generate_energy_data,
    train_runtime_models,
)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 20)
print('LUMAQ modelling environment ready')

## 2. Reproducible demonstration dataset

In [ ]:
data = generate_energy_data(rows=2400, seed=42)
print(f'Records: {len(data):,}')
print(f'Data source: {data["data_source"].unique().tolist()}')
data.head()

In [ ]:
quality = pd.DataFrame({
    'data_type': data.dtypes.astype(str),
    'missing_values': data.isna().sum(),
    'unique_values': data.nunique()
})
quality

## 3. Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(data=data, x='runtime_minutes', bins=40, ax=axes[0], color='#ff6b2c')
axes[0].set_title('Distribution of remaining runtime')
axes[0].set_xlabel('Runtime (minutes)')
sns.scatterplot(
    data=data.sample(700, random_state=42),
    x='load_power_w', y='runtime_minutes', hue='state_of_charge_pct',
    palette='viridis', alpha=0.7, ax=axes[1]
)
axes[1].set_title('Runtime decreases as load increases')
axes[1].set_xlabel('Connected load (W)')
axes[1].set_ylabel('Runtime (minutes)')
plt.tight_layout()
plt.show()

In [ ]:
summary = data.groupby('load_category').agg(
    observations=('load_power_w', 'size'),
    mean_load_w=('load_power_w', 'mean'),
    median_runtime_min=('runtime_minutes', 'median')
).round(2)
summary

## 4. Runtime prediction

In [ ]:
runtime_model, metrics, predictions = train_runtime_models(data)
metrics.round(4)

In [ ]:
best_model_name = metrics.iloc[0]['model']
prediction_column = f'prediction_{best_model_name}'

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(predictions[TARGET], predictions[prediction_column], alpha=0.45, color='#0f766e')
limit = max(predictions[TARGET].max(), predictions[prediction_column].max())
ax.plot([0, limit], [0, limit], '--', color='black', linewidth=1)
ax.set_xlabel('Actual runtime (minutes)')
ax.set_ylabel('Predicted runtime (minutes)')
ax.set_title(f'Actual and predicted runtime: {best_model_name}')
plt.tight_layout()
plt.show()

## 5. Abnormal-consumption detection

In [ ]:
anomaly_model, scored = fit_anomaly_model(data)
matrix = confusion_matrix(scored['known_anomaly'], scored['detected_anomaly'])
anomaly_metrics = pd.Series({
    'precision': precision_score(scored['known_anomaly'], scored['detected_anomaly'], zero_division=0),
    'recall': recall_score(scored['known_anomaly'], scored['detected_anomaly'], zero_division=0),
    'f1': f1_score(scored['known_anomaly'], scored['detected_anomaly'], zero_division=0),
    'flagged_records': int(scored['detected_anomaly'].sum())
})
print('Confusion matrix')
print(matrix)
anomaly_metrics

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
sample = scored.iloc[:500]
ax.plot(sample['timestamp'], sample['load_power_w'], color='#17365d', linewidth=1, label='Load')
flagged = sample[sample['detected_anomaly'] == 1]
ax.scatter(flagged['timestamp'], flagged['load_power_w'], color='#ef4444', s=35, label='Detected anomaly')
ax.set_title('LUMAQ abnormal-consumption flags')
ax.set_ylabel('Load power (W)')
ax.legend()
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## 6. Energy guidance demonstration

In [ ]:
example = pd.DataFrame([{
    'battery_voltage_v': 49.8,
    'battery_current_a': 18.0,
    'state_of_charge_pct': 31.0,
    'load_power_w': 1450.0,
    'solar_input_w': 120.0,
    'ambient_temperature_c': 31.0,
    'hour': 21.5,
    'daylight': 0,
    'load_category': 'heavy'
}])

predicted_runtime = max(0.0, float(runtime_model.predict(example[FEATURES])[0]))
guidance_record = example.iloc[0].to_dict()
guidance_record['predicted_runtime_minutes'] = predicted_runtime
guidance_record['detected_anomaly'] = 0

print(f'Predicted runtime: {predicted_runtime:.1f} minutes')
print(energy_guidance(guidance_record))

## 7. Conclusion

The prototype establishes a reproducible Python workflow for runtime prediction and abnormal-consumption detection. The evaluation results describe performance on simulated records. The physical LUMAQ gateway can supply the same input fields for subsequent calibration and field validation.